In [ ]:
!pip3 install gtts pydub langdetect

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 981.5/981.5 kB 13.9 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 98.2/98.2 kB 5.5 MB/s eta 0:00:00
  Created wheel for langdetect: filename=langdetect-1.0.9-py3-none-any.whl size=993223 sha256=acb7927a824f663f55a4693d2e5f2d4e66c98cc93890511eafba0058623c626d
  Stored in directory: /root/.cache/pip/wheels/c1/67/88/e844b5b022812e15a52e4eaa38a1e709e99f06f6639d7e3ba7
Successfully built langdetect
  Attempting uninstall: click
    Found existing installation: click 8.3.0
    Uninstalling click-8.3.0:
      Successfully uninstalled click-8.3.0


In [ ]:
from gtts import gTTS
from pydub import AudioSegment
import os
import time

/usr/local/lib/python3.12/dist-packages/pydub/utils.py:300: SyntaxWarning: invalid escape sequence '\('
  m = re.match('([su]([0-9]{1,2})p?) \(([0-9]{1,2}) bit\)$', token)
/usr/local/lib/python3.12/dist-packages/pydub/utils.py:301: SyntaxWarning: invalid escape sequence '\('
  m2 = re.match('([su]([0-9]{1,2})p?)( \(default\))?$', token)
/usr/local/lib/python3.12/dist-packages/pydub/utils.py:310: SyntaxWarning: invalid escape sequence '\('
  elif re.match('(flt)p?( \(default\))?$', token):
/usr/local/lib/python3.12/dist-packages/pydub/utils.py:314: SyntaxWarning: invalid escape sequence '\('
  elif re.match('(dbl)p?( \(default\))?$', token):


In [ ]:
# Create a folder for audio files
audio_folder = 'audio_files'
os.makedirs(audio_folder, exist_ok=True)

def create_speech_file(word, examples, file_index):
    """
    Generate audio for Italian word and sentences only (no English)
    Add 1.5 second pause between sentences
    """
    audio_segments = []

    # Generate audio for the word first
    word_audio_path = f"temp_word_{file_index}.mp3"
    tts_word = gTTS(text=word, lang='it', slow=False)
    tts_word.save(word_audio_path)
    audio_segments.append(AudioSegment.from_file(word_audio_path))

    # Add 1.5 second pause after the word
    silence = AudioSegment.silent(duration=1500)
    audio_segments.append(silence)

    # Generate audio for each Italian sentence with pauses
    for idx, sentence in enumerate(examples):
        temp_path = f"temp_sentence_{file_index}_{idx}.mp3"
        tts_sentence = gTTS(text=sentence, lang='it', slow=False)
        tts_sentence.save(temp_path)

        # Add the sentence audio
        audio_segments.append(AudioSegment.from_file(temp_path))

        # Add 1.5 second pause between sentences (but not after the last one)
        if idx < len(examples) - 1:
            audio_segments.append(silence)

        # Clean up temp file
        os.remove(temp_path)

    # Combine all audio segments
    combined_audio = sum(audio_segments)

    # Save the final audio file
    filename = os.path.join(audio_folder, f"{word}.mp3")
    combined_audio.export(filename, format="mp3")

    # Clean up word temp file
    os.remove(word_audio_path)

    print(f"Created {filename} with speech for {word}")
    return filename

def get_audio_duration(filename):
    """Check the duration of the audio file"""
    audio = AudioSegment.from_file(filename)
    return len(audio) / 1000  # Duration in seconds

# Read the Italian words and examples from the file
#/content/italian_vocabulary_batch_AI.txt
with open('/content/italian_vocabulary_with_examples.txt', 'r', encoding='utf-8') as file:
    lines = file.readlines()

# Variables to store current word and Italian examples
current_word = ""
current_italian_examples = []
files_created = 0

# Process the lines with new format
i = 0
while i < len(lines):
    line = lines[i].strip()

    if not line:
        i += 1
        continue  # Skip empty lines

    # Check if line contains word definition (has brackets)
    if '[' in line and ']' in line and '-' in line:
        # If we have a previous word, create audio for it
        if current_word and current_italian_examples:
            audio_filename = create_speech_file(
                current_word,
                current_italian_examples,
                files_created + 1
            )
            duration = get_audio_duration(audio_filename)
            print(f"Duration of {audio_filename}: {duration:.2f} seconds")

            # Wait to avoid hitting rate limit
            time.sleep(2)

        # Extract the word (before the bracket)
        current_word = line.split('[')[0].strip()
        current_italian_examples = []
        files_created += 1
        i += 1
    else:
        # This is an Italian sentence (English translation is on next line)
        if i + 1 < len(lines):
            italian_sentence = line
            # Skip the English translation (next line)
            current_italian_examples.append(italian_sentence)
            i += 2  # Skip both Italian and English lines
        else:
            i += 1

# Create audio for the last word
if current_word and current_italian_examples:
    audio_filename = create_speech_file(
        current_word,
        current_italian_examples,
        files_created + 1
    )
    duration = get_audio_duration(audio_filename)
    print(f"Duration of {audio_filename}: {duration:.2f} seconds")

print("All speech files generated!")

Created audio_files/attore.mp3 with speech for attore
Duration of audio_files/attore.mp3: 16.62 seconds
Created audio_files/promozione.mp3 with speech for promozione
Duration of audio_files/promozione.mp3: 19.28 seconds
Created audio_files/palla.mp3 with speech for palla
Duration of audio_files/palla.mp3: 15.80 seconds
Created audio_files/cliente.mp3 with speech for cliente
Duration of audio_files/cliente.mp3: 17.77 seconds
Created audio_files/fratello.mp3 with speech for fratello
Duration of audio_files/fratello.mp3: 19.23 seconds
Created audio_files/normale.mp3 with speech for normale
Duration of audio_files/normale.mp3: 17.34 seconds
Created audio_files/test.mp3 with speech for test
Duration of audio_files/test.mp3: 16.71 seconds
Created audio_files/sanità.mp3 with speech for sanità
Duration of audio_files/sanità.mp3: 18.39 seconds
Created audio_files/parco.mp3 with speech for parco
Duration of audio_files/parco.mp3: 17.29 seconds
Created audio_files/popolazione.mp3 with speech for 

In [ ]:
import zipfile
import os
def zip_audio_files(folder_name):
    zip_filename = f"{folder_name}.zip"  # Name of the zip file
    with zipfile.ZipFile(zip_filename, 'w') as zipf:
        # Walk through the folder and add files to the zip
        for root, dirs, files in os.walk(folder_name):
            for file in files:
                file_path = os.path.join(root, file)
                zipf.write(file_path, os.path.relpath(file_path, folder_name))
    return zip_filename

In [ ]:
from google.colab import files
zip_filename = zip_audio_files('audio_files')
print(f"Created zip file: {zip_filename}")
files.download(zip_filename)

Created zip file: audio_files.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
!pip install genanki==0.13.0

In [ ]:
import genanki
import os
import random
import html

# Define a unique model_id and deck_id
model_id = random.randrange(1 << 30, 1 << 31)
deck_id = random.randrange(1 << 30, 1 << 31)

# Create the Anki model
my_model = genanki.Model(
    model_id,
    'Italian Vocabulary Model',
    fields=[
        {'name': 'Italian Word'},
        {'name': 'Meaning'},
        {'name': 'Examples'},
        {'name': 'Audio'},
    ],
    templates=[
        {
            'name': 'Card 1',
            'qfmt': '''
                <div class="card">
                    <div class="question">{{Italian Word}}</div>
                </div>
            ''',  # Front side: just the Italian word
            'afmt': '''
                <div class="card">
                    <div class="question">{{FrontSide}}</div>
                    <hr id="answer">
                    <div class="answer">
                        <div class="meaning">{{Meaning}}</div>
                        <div class="examples">
                            <ul>
                                {{Examples}}
                            </ul>
                        </div>
                        <div class="audio">{{Audio}}</div>
                    </div>
                </div>
            ''',  # Back side: Meaning, examples, and audio
        },
    ],
    css="""
        .card {
            font-family: 'Arial', sans-serif;
            font-size: 20px;
            text-align: center;
            color: #333;
            background-color: #f9f9f9;
            padding: 20px;
            border-radius: 10px;
            box-shadow: 0 2px 10px rgba(0,0,0,0.1);
            margin: 20px;
        }
        .question {
            font-size: 36px;
            font-weight: bold;
            color: #2980b9;
            margin-bottom: 10px;
        }
        .answer {
            font-size: 20px;
            color: #555;
        }
        .meaning {
            font-size: 24px;
            margin: 10px 0;
            color: #2c3e50;
        }
        .examples {
            font-size: 18px;
            color: #7f8c8d;
            margin: 10px 0;
            list-style-type: none; /* Remove bullet points */
            padding: 0; /* Remove default padding */
        }
        .examples li {
            margin: 10px 0; /* Add margin between examples */
        }
        .examples li .translation {
            display: block;
            margin-left: 20px; /* Indent the translation */
            color: #34495e;
            font-style: italic;
        }
        .audio {
            font-size: 18px;
            margin-top: 20px;
        }
        a {
            color: #3498db;
            text-decoration: none;
        }
        a:hover {
            text-decoration: underline;
        }
    """
)

# Create a deck
my_deck = genanki.Deck(deck_id, '1112 Most Common Italian Words(with Audio)')

# Prepare media files list
media_files = []

# Read the Italian words and their examples from the sentences.txt file
with open('/content/italian_vocabulary_with_examples.txt', 'r', encoding='utf-8') as file:
    lines = file.readlines()

current_word = ""
current_word_type = ""
current_meaning = ""
current_examples = []

# Process the lines with new format: word [type] - meaning
i = 0
while i < len(lines):
    line = lines[i].strip()
    if not line:
        i += 1
        continue  # Skip empty lines

    # Check if line contains word definition with brackets: word [type] - meaning
    if '[' in line and ']' in line and '-' in line:
        # If there is a current word, create a note for it
        if current_word:
            # Create the audio filename pattern based on the current word
            # FIXED: Use exact filename matching instead of endswith
            audio_filename = None
            expected_filename = f"{current_word}.mp3"
            audio_path = os.path.join('/content/audio_files', expected_filename)

            if os.path.exists(audio_path):
                audio_filename = audio_path
                media_files.append(audio_filename)

                # Ensure all fields are properly encoded
                word_field = current_word.encode('utf-8').decode('utf-8')
                meaning_field = current_meaning.encode('utf-8').decode('utf-8')

                # Format examples with proper encoding
                examples_html = ""
                for example in current_examples:
                    italian_text = example[0].encode('utf-8').decode('utf-8')
                    english_text = example[1].encode('utf-8').decode('utf-8')
                    examples_html += f"<li>{html.escape(italian_text)}<span class='translation'>{html.escape(english_text)}</span></li>"

                # Create an Anki note
                my_note = genanki.Note(
                    model=my_model,
                    fields=[
                        html.escape(word_field),
                        html.escape(meaning_field),
                        examples_html,
                        f"[sound:{os.path.basename(audio_filename)}]",
                    ]
                )
                my_deck.add_note(my_note)

        # Parse new format: word [type] - meaning
        # Split by '[' to get the word
        parts = line.split('[')
        current_word = parts[0].strip()

        # Extract word type (between brackets)
        bracket_content = parts[1].split(']')
        current_word_type = bracket_content[0].strip()

        # Extract meaning (after the dash)
        after_bracket = ']'.join(bracket_content[1:])
        if '-' in after_bracket:
            meaning_part = after_bracket.split('-', 1)[1].strip()
        else:
            meaning_part = after_bracket.strip()
        current_meaning = f"({current_word_type}) {meaning_part}"

        current_examples = []  # Reset examples list
        i += 1  # Move to next line
    else:
        # If it's an Italian sentence, the next line should be its English translation
        if i + 1 < len(lines):
            italian_sentence = line
            english_translation = lines[i + 1].strip()  # Get the next line as translation
            current_examples.append((italian_sentence, english_translation))  # Add the pair
            i += 2  # Skip the next line (English translation)
        else:
            i += 1  # Just move to the next line if no translation is found

# Create an Anki note for the last word
if current_word:
    # FIXED: Use exact filename matching instead of endswith
    audio_filename = None
    expected_filename = f"{current_word}.mp3"
    audio_path = os.path.join('/content/audio_files', expected_filename)

    if os.path.exists(audio_path):
        audio_filename = audio_path
        media_files.append(audio_filename)

        # Ensure all fields are properly encoded
        word_field = current_word.encode('utf-8').decode('utf-8')
        meaning_field = current_meaning.encode('utf-8').decode('utf-8')

        # Format examples with proper encoding
        examples_html = ""
        for example in current_examples:
            italian_text = example[0].encode('utf-8').decode('utf-8')
            english_text = example[1].encode('utf-8').decode('utf-8')
            examples_html += f"<li>{html.escape(italian_text)}<span class='translation'>{html.escape(english_text)}</span></li>"

        my_note = genanki.Note(
            model=my_model,
            fields=[
                html.escape(word_field),
                html.escape(meaning_field),
                examples_html,
                f"[sound:{os.path.basename(audio_filename)}]",
            ]
        )
        my_deck.add_note(my_note)

# Create an Anki package with media files
my_package = genanki.Package(my_deck)
my_package.media_files = media_files
my_package.write_to_file('1112_most_common_italian_vocabs.apkg')  # Write the deck to a file

# Download the Anki deck
from google.colab import files
files.download('1112_most_common_italian_vocabs.apkg')  # Download the generated Anki deck

print("Anki deck created successfully!")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Anki deck created successfully!
